# Huấn luyện XGBoost cho matching CV ↔ JD (IT)

**Mục tiêu:** dự đoán điểm phù hợp (nhãn 0, 1, 3, 4) từ 11 feature A1–C2.

**Thiết kế đánh giá (quan trọng):**
- Chia theo **JD** (leave-one-JD-out): mỗi lần giữ lại 1 JD để test, huấn luyện trên 9 JD còn lại.
- Chỉ **test trên dòng thật** (`is_synthetic = 0`). Dòng tổng hợp chỉ dùng để train.
- So sánh 3 cách: (a) điểm prior weighted-sum, (b) XGBoost chỉ train dữ liệu thật, (c) XGBoost train thật + tổng hợp.

**Chuẩn bị:** upload `features_300.csv` và `labels_300.csv` (bước 2 sẽ hiện nút upload).

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, cohen_kappa_score, confusion_matrix

print('xgboost', xgb.__version__)
SEED = 42

## 1. Nạp dữ liệu

In [ ]:
# Nếu chạy trên Colab và chưa có file thì hiện hộp upload
if not (os.path.exists('features_300.csv') and os.path.exists('labels_300.csv')):
    from google.colab import files
    files.upload()   # chọn cả 2 file: features_300.csv và labels_300.csv

feat = pd.read_csv('features_300.csv')
lab  = pd.read_csv('labels_300.csv')

# label đã có trong features; chỉ lấy thêm cột is_synthetic từ file nhãn
df = feat.merge(lab[['jd_id', 'cv_id', 'is_synthetic']], on=['jd_id', 'cv_id'], validate='1:1')

print(df.shape)
print('Dòng thật / tổng hợp:', df.is_synthetic.value_counts().sort_index().to_dict())
print('Phân bố nhãn:', df.label.value_counts().sort_index().to_dict())
df.head()

## 2. Chuẩn bị feature

- Dùng trực tiếp 11 feature A1–C2.
- Giữ nguyên `NaN` (XGBoost xử lý giá trị thiếu tự nhiên, không cần cột chỉ báo).

In [ ]:
X_COLS = ['A1','A2','A3','B1','B2','B3','B4','B5','B6','C1','C2']

# Ràng buộc đơn điệu: feature tăng thì điểm không được giảm.
# KHÔNG áp cho B2, B3 (quá nhiều kinh nghiệm/quá cấp bậc lại làm điểm giảm),
# B5 và C2.
MONO = {'A1':1,'A2':1,'A3':1,'B1':1,'B4':1,'B6':1,'C1':1}
mono_tuple = tuple(MONO.get(c, 0) for c in X_COLS)
print(dict(zip(X_COLS, mono_tuple)))

## 3. Hàm hỗ trợ

Dùng **hồi quy thứ tự** (dự đoán số thực trên thang 0–4, giữ khoảng cách 0/1/3/4) thay vì phân lớp, vì:
nhãn có thứ tự, và ràng buộc đơn điệu chỉ có nghĩa với đầu ra một chiều (với phân lớp nhiều lớp, ràng buộc này mâu thuẫn giữa các lớp).

Sau model áp **cổng must-have** (A1 < 0.5 → trần 50 điểm, 0.5–0.7 → trần 70, thang 0–100).

In [ ]:
VALID = np.array([0, 1, 3, 4])

def snap(p):                      # đưa điểm thực về nhãn hợp lệ gần nhất
    p = np.asarray(p)
    return VALID[np.abs(p[:, None] - VALID[None, :]).argmin(axis=1)]

def apply_gate(pred04, a1):       # pred04: thang 0-4 -> điểm 0-100, có trần theo A1
    score = np.clip(np.asarray(pred04), 0, 4) * 25
    cap = np.where(a1 < 0.5, 50, np.where(a1 < 0.7, 70, 100))
    return np.minimum(score, cap)

def make_model(**kw):
    p = dict(n_estimators=300, learning_rate=0.05, max_depth=3, min_child_weight=2,
             subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
             objective='reg:squarederror', tree_method='hist',
             monotone_constraints=mono_tuple, random_state=SEED)
    p.update(kw)
    return xgb.XGBRegressor(**p)

def leave_one_jd_out(use_syn=True, syn_weight=1.0, **model_kw):
    """Train trên 9 JD (kèm/không kèm dòng tổng hợp), test trên dòng THẬT của JD còn lại."""
    pred = pd.Series(np.nan, index=df.index)
    for jd in sorted(df.jd_id.unique()):
        te = df[(df.jd_id == jd) & (df.is_synthetic == 0)]
        tr = df[df.jd_id != jd]
        if not use_syn:
            tr = tr[tr.is_synthetic == 0]
        w = np.where(tr.is_synthetic == 1, syn_weight, 1.0)
        m = make_model(**model_kw)
        m.fit(tr[X_COLS], tr.label, sample_weight=w)
        pred[te.index] = m.predict(te[X_COLS])
    return pred.dropna()

def report(pred, name):
    y = df.loc[pred.index, 'label'].values
    ps = snap(pred.values)
    return {
        'model': name,
        'MAE (0-4)': round(mean_absolute_error(y, pred.values), 3),
        'Spearman': round(spearmanr(y, pred.values)[0], 3),
        'QWK': round(cohen_kappa_score(y, ps, weights='quadratic'), 3),
        'Accuracy': round((ps == y).mean(), 3),
    }

## 4. Đánh giá leave-one-JD-out và so sánh với baseline

In [ ]:
# (a) Baseline: weighted-sum prior (NaN -> bỏ ra và chuẩn hoá lại trọng số)
W = dict(A1=.25,A2=.05,A3=.08,B1=.10,B2=.10,B3=.07,B4=.15,B5=.08,B6=.04,C1=.05,C2=.03)
def prior_score(r):
    num = den = 0
    for k, w in W.items():
        if not np.isnan(r[k]): num += w * r[k]; den += w
    return num / den
real = df[df.is_synthetic == 0]
prior = real.apply(prior_score, axis=1)
base_row = {'model': '(a) Prior weighted-sum', 'MAE (0-4)': '-', 'QWK': '-', 'Accuracy': '-',
            'Spearman': round(spearmanr(real.label, prior)[0], 3)}

pred_real = leave_one_jd_out(use_syn=False)
pred_all  = leave_one_jd_out(use_syn=True)

res = pd.DataFrame([base_row,
                    report(pred_real, '(b) XGB - chỉ dữ liệu thật'),
                    report(pred_all,  '(c) XGB - thật + tổng hợp')])
res

In [ ]:
# Chi tiết model (c): ma trận nhầm lẫn và kết quả theo từng JD
y_true = df.loc[pred_all.index, 'label'].values
y_hat  = snap(pred_all.values)
print('Ma trận nhầm lẫn (hàng = nhãn thật, cột = dự đoán; thứ tự 0,1,3,4)')
print(pd.DataFrame(confusion_matrix(y_true, y_hat, labels=VALID),
                   index=[f'thật {v}' for v in VALID], columns=[f'dđ {v}' for v in VALID]))

per_jd = (pd.DataFrame({'jd': df.loc[pred_all.index, 'jd_id'], 'y': y_true, 'p': pred_all.values})
            .groupby('jd').apply(lambda g: pd.Series({
                'Spearman': spearmanr(g.y, g.p)[0],
                'MAE': mean_absolute_error(g.y, g.p)}), include_groups=False).round(3))
per_jd

## 5. (Tuỳ chọn) Tinh chỉnh siêu tham số

Chỉ có 100 dòng thật để test nên **đừng tinh chỉnh quá nhiều**, dễ overfit vào đúng 100 dòng này. Chọn cấu hình đơn giản và ổn định, không nhất thiết cấu hình điểm cao nhất.

In [ ]:
import itertools
rows = []
for depth, n_est, lr in itertools.product([2, 3, 4], [150, 300, 500], [0.03, 0.05]):
    p = leave_one_jd_out(use_syn=True, max_depth=depth, n_estimators=n_est, learning_rate=lr)
    rows.append({'max_depth': depth, 'n_estimators': n_est, 'learning_rate': lr, **report(p, '')})
grid = pd.DataFrame(rows).drop(columns='model').sort_values('MAE (0-4)')
grid.head(10)

## 6. Huấn luyện model cuối trên toàn bộ 300 dòng và xem feature importance

In [ ]:
# Điền cấu hình bạn chọn từ bước 5 (mặc định giữ như dưới)
FINAL = dict(max_depth=3, n_estimators=300, learning_rate=0.05)

final = make_model(**FINAL)
final.fit(df[X_COLS], df.label)

imp = pd.Series(final.get_booster().get_score(importance_type='gain')).reindex(X_COLS).fillna(0).sort_values()
imp.plot.barh(figsize=(6, 4), title='Feature importance (gain)')
plt.tight_layout(); plt.show()

## 7. Lưu model và dự đoán cặp CV–JD mới

In [ ]:
final.save_model('cv_jd_xgb.json')          # tải về từ thanh Files bên trái của Colab

def predict_match(feat_dict):
    """feat_dict: dict các feature A1..C2 (dùng None hoặc np.nan cho giá trị thiếu / không áp dụng)."""
    row = {k: (np.nan if feat_dict.get(k) is None else feat_dict[k]) for k in X_COLS}
    x = pd.DataFrame([row])[X_COLS]
    p = float(final.predict(x)[0])
    score = float(apply_gate([p], np.array([row['A1']]))[0])
    gate = 'passed' if row['A1'] >= 0.7 else ('cap 70' if row['A1'] >= 0.5 else 'cap 50')
    return {'pred_0_4': round(p, 2), 'score_0_100': round(score), 'gate': gate}

# Ví dụ: vector trong tài liệu (B5 thiếu, C2 không áp dụng)
example = dict(A1=0.74, A2=0.0, A3=0.67, B1=1.0, B2=0.67, B3=0.70,
               B4=0.70, B5=None, B6=0.80, C1=0.88, C2=None)
predict_match(example)